# Session 3 — DRL Portfolio Optimisation + Markowitz+ML

**Course:** AI in Investment Strategies  
**Instructor:** Robert Ślepaczuk, WNE University of Warsaw

## Objectives
1. Markowitz baseline + Ledoit-Wolf shrinkage + ML-enhanced $\mu$
2. Custom gymnasium portfolio environment with transaction costs
3. SAC agent training (stable-baselines3) — Kashif & Ślepaczuk (arXiv:2605.17307, 2026)
4. Reward function design: raw P&L vs. Sharpe-based vs. Sortino-based
5. Full performance comparison and risk analysis

### Key references
- Kashif & Ślepaczuk (2026) — DRL portfolio, *arXiv:2605.17307*
- Ślusarczyk & Ślepaczuk (2025) — ML Markowitz, *Journal of Big Data* 12(1)
- Lebiedź & Ślepaczuk (2026) — Multi-pair crypto DRL, *arXiv:2606.04574*

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import yfinance as yf
import gymnasium as gym
from gymnasium import spaces
from scipy.optimize import minimize
import warnings
warnings.filterwarnings('ignore')

plt.style.use('seaborn-v0_8-whitegrid')
COLORS = {'naive': '#888888', 'markowitz': '#003366', 'ml_mw': '#228B22',
          'drl': '#c41230', 'bh': '#aaaaaa'}
TICKERS = ['SPY', 'TLT', 'GLD', 'QQQ', 'EEM', 'VNQ']

## 1. Data

In [ ]:
raw    = yf.download(TICKERS, start='2010-01-01', end='2024-12-31', auto_adjust=True)['Close'].dropna()
returns = raw.pct_change().dropna()
n_assets = len(TICKERS)

print(f'Assets: {TICKERS}')
print(f'Period: {returns.index[0].date()} – {returns.index[-1].date()}')
print(f'Shape:  {returns.shape}')

# Summary stats
ann_ret = returns.mean() * 252
ann_vol = returns.std() * np.sqrt(252)
corr    = returns.corr()
pd.DataFrame({'ARC (approx)': ann_ret.map('{:.1%}'.format),
              'Ann Vol':        ann_vol.map('{:.1%}'.format)})

## 2. Performance Utilities

In [ ]:
def sharpe(r, ann=252):
    return r.mean() / (r.std() + 1e-12) * np.sqrt(ann)

def sortino(r, ann=252):
    downside = r.clip(upper=0).std() + 1e-12
    return r.mean() / downside * np.sqrt(ann)

def max_drawdown(r):
    w = (1 + r).cumprod()
    return ((w - w.cummax()) / w.cummax()).min()

def cagr(r, ann=252):
    return (1 + r).prod() ** (ann / len(r)) - 1

def metrics(r):
    return {
        'ARC':    f'{cagr(r):.1%}',
        'Sharpe':  round(sharpe(r), 2),
        'Sortino': round(sortino(r), 2),
        'MD':     f'{max_drawdown(r):.1%}',
        'Calmar':  round(cagr(r) / abs(max_drawdown(r)), 2),
    }

## 3. Markowitz Optimisation with Ledoit-Wolf Shrinkage

Replaces historical $\hat\Sigma$ with Ledoit-Wolf regularised estimator to reduce estimation error.

In [ ]:
from sklearn.covariance import LedoitWolf

def markowitz_max_sharpe(mu: np.ndarray, cov: np.ndarray,
                          rf: float = 0.0, long_only: bool = True) -> np.ndarray:
    n = len(mu)

    def neg_sharpe(w):
        r_p = mu @ w
        v_p = np.sqrt(w @ cov @ w)
        return -(r_p - rf) / (v_p + 1e-12)

    bounds = [(0, 1) if long_only else (-0.5, 1)] * n
    result = minimize(
        neg_sharpe, x0=np.ones(n) / n,
        method='SLSQP',
        bounds=bounds,
        constraints={'type': 'eq', 'fun': lambda w: w.sum() - 1},
        options={'maxiter': 500, 'ftol': 1e-9}
    )
    return result.x if result.success else np.ones(n) / n


def run_markowitz_wfo(returns: pd.DataFrame, is_days: int = 252, oos_days: int = 63,
                       tc: float = 0.001, use_lw: bool = True) -> pd.Series:
    oos, prev_w = [], None
    for start in range(is_days, len(returns) - oos_days, oos_days):
        r_is  = returns.iloc[start - is_days : start]
        r_oos = returns.iloc[start           : start + oos_days]

        mu  = r_is.mean().values
        cov = (LedoitWolf().fit(r_is).covariance_ if use_lw
               else r_is.cov().values)

        w_opt = markowitz_max_sharpe(mu, cov)

        r_period = r_oos.values @ w_opt
        if prev_w is not None:
            r_period[0] -= tc * np.abs(w_opt - prev_w).sum() / 2
        prev_w = w_opt

        oos.append(pd.Series(r_period, index=r_oos.index))

    return pd.concat(oos) if oos else pd.Series(dtype=float)


print('Running Markowitz WFO (vanilla and LW)...')
r_mw_plain = run_markowitz_wfo(returns, use_lw=False)
r_mw_lw    = run_markowitz_wfo(returns, use_lw=True)
r_naive    = (returns @ np.ones(n_assets) / n_assets)
r_naive    = r_naive.reindex(r_mw_lw.index)

pd.DataFrame({n: metrics(r) for n, r in [
    ('1/N Naive',  r_naive),
    ('MV (plain)', r_mw_plain),
    ('MV + LW',    r_mw_lw),
]}).T

## Interpretation: Markowitz — Results and Limitations

**Comparison results (typical for a 6-asset portfolio):**

| Strategy | Sharpe | MD | Notes |
|-----------|--------|-----|-------|
| 1/N Naive | ~0.55 | ~38% | Simple, but surprisingly good (DeMiguel et al., 2009) |
| MV (plain) | ~0.48 | ~51% | Worse MD! Extreme weights destabilise the portfolio |
| MV + LW | ~0.62 | ~32% | Shrinkage improves weight stability |

**Why CAN Markowitz without shrinkage be worse than 1/N?**
- Estimating μ (expected returns) is very noisy.
- The error in μ propagates into the weights → "error maximiser" rather than "return maximiser".
- 1/N avoids this error: it ignores μ and limits concentration.

**How does Ledoit-Wolf (LW) work?**
Regularised covariance matrix:
```
Σ_LW = (1 − δ)·Σ̂ + δ·σ̄²·I
```
- δ = optimal shrinkage coefficient (derived analytically)
- Shifts Σ̂ toward a diagonal matrix (assumes no correlation)
- Reduces the estimation of extreme weights → smaller MD in OOS

**Key observation:** Markowitz is sensitive to the quality of its inputs (μ, Σ).
The ML-enhanced approach (Section 4) improves μ → better OOS results.

## 4. ML-Enhanced Markowitz

Replace historical $\hat\mu$ with LSTM forecasts of next-period returns.  
Ślusarczyk & Ślepaczuk (Journal of Big Data, 2025)

In [ ]:
import torch, torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
from sklearn.preprocessing import StandardScaler

# Reproducibility for the ML-Markowitz LSTM (matches SAC seeding in Sec. 6)
np.random.seed(42)
torch.manual_seed(42)

import os
# Urzadzenie: CUDA gdy dostepna. MPS (Apple Silicon) TYLKO na wyrazne zyczenie,
# bo liczy w innej kolejnosci zmiennoprzecinkowej niz CPU i te same komorki daja
# nieco inne wyniki - a material dydaktyczny ma dawac te same liczby na kazdej
# maszynie. Wlaczenie: ustaw COURSE_USE_MPS=1 przed uruchomieniem Jupytera.
DEVICE = ('cuda' if torch.cuda.is_available()
          else 'mps' if os.environ.get('COURSE_USE_MPS') and torch.backends.mps.is_available()
          else 'cpu')

class MultiOutputLSTM(nn.Module):
    """LSTM that forecasts returns for all n_assets simultaneously."""
    def __init__(self, input_size: int, n_out: int, hidden: int = 64, layers: int = 1):
        super().__init__()
        self.lstm = nn.LSTM(input_size, hidden, layers, batch_first=True, dropout=0.1)
        self.head = nn.Linear(hidden, n_out)

    def forward(self, x):
        out, _ = self.lstm(x)
        return self.head(out[:, -1, :])


def make_ml_features(returns: pd.DataFrame, windows=(5, 21)) -> np.ndarray:
    feats = []
    for w in windows:
        feats.append(returns.rolling(w).mean())
        feats.append(returns.rolling(w).std())
    return pd.concat(feats, axis=1).dropna().values


def run_ml_markowitz_wfo(returns: pd.DataFrame, is_days: int = 252,
                          oos_days: int = 63, epochs: int = 20,
                          tc: float = 0.001) -> pd.Series:
    X_all = make_ml_features(returns)
    y_all = returns.values[returns.shape[1]*4:]  # align after feature creation
    feat_idx_start = returns.shape[0] - len(X_all)
    ret_dates = returns.index[feat_idx_start:]
    SEQ = 10

    oos, prev_w = [], None

    for start in range(is_days, len(X_all) - oos_days, oos_days):
        X_tr = X_all[start - is_days : start].astype(np.float32)
        y_tr = y_all[start - is_days : start].astype(np.float32)

        scaler = StandardScaler().fit(X_tr)
        Xn = scaler.transform(X_tr)

        # Build sequences
        Xs = np.array([Xn[i - SEQ:i] for i in range(SEQ, len(Xn))], dtype=np.float32)
        ys = y_tr[SEQ:]

        model = MultiOutputLSTM(Xs.shape[2], n_out=n_assets).to(DEVICE)
        opt   = torch.optim.Adam(model.parameters(), lr=1e-3)
        ds    = TensorDataset(torch.from_numpy(Xs), torch.from_numpy(ys))
        dl    = DataLoader(ds, batch_size=32, shuffle=True)

        model.train()
        for _ in range(epochs):
            for xb, yb in dl:
                xb, yb = xb.to(DEVICE), yb.to(DEVICE)
                loss = nn.MSELoss()(model(xb), yb)
                opt.zero_grad(); loss.backward(); opt.step()

        # Predict mu for next period (use last SEQ-length window)
        X_pred = scaler.transform(X_all[start - SEQ : start].astype(np.float32))
        X_pred_t = torch.from_numpy(X_pred[np.newaxis]).to(DEVICE)
        model.eval()
        with torch.no_grad():
            mu_ml = model(X_pred_t).cpu().numpy().flatten()

        # LW covariance on IS data
        cov_lw = LedoitWolf().fit(y_tr).covariance_
        w_opt  = markowitz_max_sharpe(mu_ml, cov_lw)

        r_oos = returns.iloc[feat_idx_start + start : feat_idx_start + start + oos_days]
        r_period = r_oos.values @ w_opt
        if prev_w is not None:
            r_period[0] -= tc * np.abs(w_opt - prev_w).sum() / 2
        prev_w = w_opt
        oos.append(pd.Series(r_period, index=r_oos.index))

    return pd.concat(oos) if oos else pd.Series(dtype=float)


print('Training ML-Markowitz WFO (takes a few minutes)...')
r_ml_mw = run_ml_markowitz_wfo(returns, epochs=15)
print(f'ML-Markowitz Sharpe: {sharpe(r_ml_mw):.2f}')

## Interpretation: ML-Enhanced Markowitz

**Idea:** Replace the historical μ with LSTM forecasts.
```
Classical:  w* = f(μ̂_historical, Σ̂_LW)
ML version: w* = f(μ̂_LSTM, Σ̂_LW)
```

**Why does it work?**
- The LSTM learns patterns of momentum, mean-reversion and volatility clustering.
- It forecasts μ better than the historical mean, especially in trends.
- LW still stabilises the weights (reduces the risk of extreme positions).

**Limitations of the approach:**
- Computational complexity: the LSTM is trained from scratch in every WFO window.
- Risk of overfitting when the IS window is small (< 252 days).
- The μ forecast by the LSTM is a 1-day return → Markowitz assumes a static investment.
  In practice rebalancing must be frequent → transaction costs increase.

**When does ML-Markowitz lose?**
- In regime-shift regimes (e.g. 2008, 2020): the LSTM has not seen such data in IS.
- When assets are strongly correlated (correlations tend to 1 in a crisis) → diversification vanishes.
- Recommendation: add an MD circuit-breaker (see Section 9).

## 5. Portfolio Environment (Gymnasium)

In [ ]:
class PortfolioEnv(gym.Env):
    """
    Gymnasium environment for multi-asset portfolio allocation.
    Supports three reward modes: 'pnl', 'sharpe', 'sortino'.

    Based on: Kashif & Slepaczuk (arXiv:2605.17307, 2026)
    """

    def __init__(self, returns: np.ndarray, lookback: int = 20,
                 tc: float = 0.001, reward_mode: str = 'sharpe'):
        super().__init__()
        self.returns     = returns.astype(np.float32)
        self.lookback    = lookback
        self.tc          = tc
        self.reward_mode = reward_mode
        self.n           = returns.shape[1]

        obs_dim = self.n * lookback + self.n
        self.observation_space = spaces.Box(-np.inf, np.inf, (obs_dim,), np.float32)
        self.action_space      = spaces.Box( 0.0,    1.0, (self.n,), np.float32)

    def reset(self, seed=None):
        super().reset(seed=seed)
        self.t       = self.lookback
        self.w       = np.ones(self.n, dtype=np.float32) / self.n
        self._hist   = []
        return self._obs(), {}

    def _obs(self):
        past = self.returns[self.t - self.lookback : self.t].flatten()
        return np.concatenate([past, self.w])

    def step(self, action):
        # Map action to probability simplex
        a   = np.exp(action - action.max())
        w_new = (a / a.sum()).astype(np.float32)

        r_t       = self.returns[self.t]
        port_ret  = float(np.dot(w_new, r_t))
        turnover  = float(np.abs(w_new - self.w).sum()) / 2.0
        net_ret   = port_ret - self.tc * turnover

        self._hist.append(net_ret)
        self.w = w_new
        self.t += 1
        done = self.t >= len(self.returns)

        reward = self._compute_reward(net_ret)
        info   = {'net_return': net_ret, 'turnover': turnover}
        return self._obs() if not done else self._obs(), reward, done, False, info

    def _compute_reward(self, net_ret: float) -> float:
        if self.reward_mode == 'pnl':
            return net_ret
        if len(self._hist) < 10:
            return net_ret
        h = np.array(self._hist[-20:])
        std = h.std() + 1e-8
        if self.reward_mode == 'sharpe':
            return float(h.mean() / std * np.sqrt(252))
        if self.reward_mode == 'sortino':
            down_std = h.clip(max=0).std() + 1e-8
            return float(h.mean() / down_std * np.sqrt(252))
        return net_ret


# Quick env check
env = PortfolioEnv(returns.values, reward_mode='sharpe')
obs, _ = env.reset()
print(f'Obs: {obs.shape} | Action: {env.action_space.shape}')
print(f'Reward mode: {env.reward_mode}')

## Interpretation: The MDP Environment — Key Design Decisions

**State Space:**
```python
obs = concat(past_returns[t-lookback:t].flatten(), current_weights)
```
- Historical returns: information about momentum/mean-reversion
- Current weights: the agent knows the cost of changing them (transaction costs)
- Lookback = 20 days: a trade-off between memory and observation dimensionality

**Action Space:**
```python
action ∈ [0, 1]^n  →  softmax  →  weights summing to 1
```
- We use log-softmax instead of direct weights: it prevents zero weights
- No short selling (long-only) → a realistic constraint for funds

**Reward Function — the most important decision:**
- `pnl`: simple, but high variance → unstable training
- `sharpe`: rolling Sharpe → the agent learns the return/risk balance
- `sortino`: favours strategies with low downside risk

**Why expose the return through softmax rather than sigmoid?**
- Sigmoid: weights may not sum to 1 → requires normalisation.
- Log-softmax: natively guarantees the simplex {w ≥ 0, Σw = 1}.
- Without an explicit softmax: backprop gradients can be unstable at boundary weights.

**Potential environment problems (reward hacking):**
- The agent may stay in cash (all weights equal) → reward ≈ 0 is "safe"
- Fix: add a penalty for low activity (`penalty_flat = max(0, 0.2 - w.max())`)

## 6. DRL Training (SAC)

SAC (Soft Actor-Critic) with entropy regularisation — preferred for continuous action spaces.

In [ ]:
try:
    from stable_baselines3 import SAC
    from stable_baselines3.common.utils import set_random_seed
    SB3 = True
except ImportError:
    SB3 = False
    print('stable-baselines3 not installed; install with: pip install stable-baselines3')

# Reproducibility: seed all RNGs so SAC training is deterministic across runs.
SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)
if SB3:
    set_random_seed(SEED)


def run_drl_wfo(returns: pd.DataFrame, is_days: int = 756, oos_days: int = 252,
                 timesteps: int = 5_000, reward_mode: str = 'sharpe') -> pd.Series:
    if not SB3:
        return pd.Series(dtype=float)

    oos_all = []
    for start in range(is_days, len(returns) - oos_days, oos_days):
        r_is  = returns.iloc[start - is_days : start].values
        r_oos = returns.iloc[start           : start + oos_days]

        train_env = PortfolioEnv(r_is, reward_mode=reward_mode)
        model = SAC(
            'MlpPolicy', train_env, verbose=0, seed=SEED,
            learning_rate=3e-4, buffer_size=20_000,
            batch_size=256, ent_coef='auto',
            policy_kwargs={'net_arch': [128, 64]},
        )
        model.learn(total_timesteps=timesteps)

        eval_env = PortfolioEnv(r_oos.values, reward_mode='pnl')  # eval on raw P&L
        obs, _   = eval_env.reset()
        done     = False
        episode  = []
        while not done:
            action, _ = model.predict(obs, deterministic=True)
            obs, r, done, _, info = eval_env.step(action)
            episode.append(info['net_return'])

        oos_all.append(pd.Series(episode, index=r_oos.index[:len(episode)]))
        sr = np.mean(episode) / (np.std(episode) + 1e-12) * np.sqrt(252)
        print(f'OOS {r_oos.index[0].date()}: SR={sr:.2f}')

    return pd.concat(oos_all) if oos_all else pd.Series(dtype=float)


# Cache SAC results so they can be shown later without re-training (~30 min/run).
# Set USE_CACHE = False to force a fresh run.
import os, pickle
from pathlib import Path
# Resolve cache to <project-root>/reports regardless of working directory,
# so results are found whether the notebook runs from repo root or notebooks/.
_cwd = Path.cwd()
_root = next((p for p in [_cwd, *_cwd.parents] if (p / 'pyproject.toml').exists()), _cwd)
CACHE_DIR = str(_root / 'reports'); os.makedirs(CACHE_DIR, exist_ok=True)
DRL_CACHE = os.path.join(CACHE_DIR, 'drl_results.pkl')
USE_CACHE = True

print('Training DRL agent...')
if USE_CACHE and os.path.exists(DRL_CACHE):
    with open(DRL_CACHE, 'rb') as fh:
        r_drl = pickle.load(fh)
    print(f'Loaded cached DRL results ({len(r_drl)} OOS days) from {DRL_CACHE}')
else:
    r_drl = run_drl_wfo(returns, reward_mode='sharpe')
    with open(DRL_CACHE, 'wb') as fh:
        pickle.dump(r_drl, fh)
    print(f'Saved DRL results to {DRL_CACHE}')

## Interpretation: SAC vs PPO — When to Use Which?

**Proximal Policy Optimisation (PPO):**
- On-policy: learns from the current policy → slower (requires fresh data)
- Stable: clipping prevents overly large policy updates
- Good for: discrete environments or when abundant data is cheap

**Soft Actor-Critic (SAC):**
- Off-policy: learns from experience in a replay buffer → faster
- Entropy: H(π) regularisation → enforces exploration; avoids local minima
- Good for: continuous action spaces (portfolio weights ∈ ℝⁿ) → **our case**

**Why is SAC preferred for portfolio allocation?**
| Criterion | PPO | SAC |
|-----------|-----|-----|
| Action space | discrete/continuous | continuous ✓ |
| Sample efficiency | low | high ✓ |
| Exploration | implicit | entropy ✓ |
| Stability | high ✓ | moderate |

**SAC hyperparameters to tune:**
- `ent_coef`: entropy coefficient; higher = more exploration
- `net_arch`: actor/critic network size; [128, 64] = a good start
- `buffer_size`: replay buffer size; larger = better, but more RAM
- `learning_rate`: 3e-4 is standard; reduce it if training is unstable
- `timesteps`: at least 50k for meaningful results; 200k+ for stabilisation

**Monitoring training:**
- `ep_rew_mean` should rise (or stabilise) → if it falls = a problem
- An overly large `ent_coef_loss` = the agent explores pointlessly; reduce ent_coef

## 7. Full Comparison

In [ ]:
common_idx = r_mw_lw.index

strategies = [
    ('1/N Naive',   r_naive.reindex(common_idx), COLORS['naive']),
    ('MV (plain)',  r_mw_plain.reindex(common_idx), '#4477AA'),
    ('MV + LW',     r_mw_lw.reindex(common_idx), COLORS['markowitz']),
    ('ML + LW',     r_ml_mw.reindex(common_idx).fillna(0), COLORS['ml_mw']),
]

if len(r_drl) > 10:
    strategies.append(('SAC-DRL', r_drl.reindex(common_idx).fillna(0), COLORS['drl']))

# Performance table
summary = pd.DataFrame({n: metrics(r) for n, r, _ in strategies}).T
print('\n--- Performance Summary (OOS) ---')
print(summary.to_string())

fig = plt.figure(figsize=(16, 10))
gs  = gridspec.GridSpec(2, 2, figure=fig, hspace=0.38, wspace=0.3)

# Equity curves
ax0 = fig.add_subplot(gs[0, :])
for name, r, c in strategies:
    r_filled = r.fillna(0)
    (1 + r_filled).cumprod().plot(ax=ax0, label=name, color=c, linewidth=1.6)
ax0.set_title('OOS Portfolio Equity Curves', fontsize=12)
ax0.set_ylabel('Cumulative Return')
ax0.legend(fontsize=9)

# Sharpe comparison
ax1 = fig.add_subplot(gs[1, 0])
names  = [s[0] for s in strategies]
srs    = [metrics(s[1].fillna(0))['Sharpe'] for s in strategies]
colors = [s[2] for s in strategies]
ax1.bar(names, srs, color=colors, alpha=0.85, edgecolor='white')
ax1.axhline(0, color='black', lw=0.5)
ax1.set_title('OOS Sharpe Ratio', fontsize=11)
ax1.set_ylabel('Sharpe')
ax1.tick_params(axis='x', rotation=20)

# Drawdown
ax2 = fig.add_subplot(gs[1, 1])
for name, r, c in strategies:
    r_f = r.fillna(0)
    w   = (1 + r_f).cumprod()
    dd  = (w - w.cummax()) / w.cummax()
    dd.plot(ax=ax2, label=name, color=c, linewidth=1.3, alpha=0.8)
ax2.fill_between(ax2.get_lines()[0].get_xdata(), ax2.get_lines()[0].get_ydata(), 0,
                  alpha=0.05, color='black')
ax2.set_title('Drawdown Comparison', fontsize=11)
ax2.set_ylabel('Drawdown')
ax2.legend(fontsize=8)

plt.savefig('../data/session3_portfolio_comparison.png', dpi=150, bbox_inches='tight')
plt.show()
print('Figure saved.')

## Interpretation of Results: Comparison of Portfolio Strategies

**How to read this results table:**

**ARC** (Annualized Return Compounded): nominal annual return
- Does not account for risk → always interpret it together with MD and Sharpe

**Sharpe Ratio** = (ARC - Rf) / ASD
- < 0.5: weak (hard to justify management fees)
- 0.5–1.0: acceptable
- 1.0–2.0: good
- > 2.0: excellent / suspicious (check costs and overfitting)

**Sortino Ratio** = ARC / downside_std × √252
- Better than Sharpe when a strategy has an asymmetric return profile (options, DRL)
- Favours strategies that have good gains but small losses

**MD** (Maximum Drawdown): the deepest decline from a peak
- Hedge fund: MD > 20% → clients withdraw capital
- Retail: MD > 30% → psychologically hard to sustain

**Measured hierarchy (Sharpe OOS, this run, seed=42):**
1. **1/N Naive → 0.77** — the best! The simple equal-weight benchmark wins (DeMiguel et al., 2009).
2. MV (plain) / MV + LW → 0.69 — classical Markowitz, decent.
3. **SAC-DRL → 0.46** — undertrained (5,000 steps/window).
4. **ML + LW → 0.34** — the weakest in this run.

**Why do complex models lose (an honest lesson):**
- Both "intelligent" approaches (SAC-DRL and ML-Markowitz with LSTM) lose to the simple 1/N and classical MV — complexity does not guarantee an edge.
- **SAC:** 5,000 steps/window is too few (DRL literature: 10⁵–10⁶). An undertrained agent does not extract a complex allocation policy.
- **ML-Markowitz:** the μ forecast from the LSTM is very sensitive to the seed — with a different seed the Sharpe can jump (~0.7). This instability is a warning: a seed-dependent model is a risk, not an edge.
- **Conclusion:** 1/N is a notoriously hard benchmark (DeMiguel et al., 2009). The value of AI in a portfolio only appears with careful tuning, a larger compute budget and hard risk management (Section 9).

> **Pedagogical note:** these figures concern an *undertrained* SAC (5k steps) and a single realisation of ML-Markowitz (seed=42), chosen for speed in class. A full picture would require training for 200k+ steps on a GPU and averaging over many seeds.

**Why does 1/N beat classical Markowitz (DeMiguel et al., 2009)?**
Markowitz needs at least N² / 2 parameters to estimate Σ.
With N=6 assets and 252 IS days → large estimation error → worse OOS results.

## 8. Reward Function Experiment

In [ ]:
if SB3 and len(r_drl) > 10:
    print('Comparing reward modes (pnl vs. sharpe vs. sortino)...')
    REWARD_CACHE = os.path.join(CACHE_DIR, 'drl_reward_results.pkl')
    if USE_CACHE and os.path.exists(REWARD_CACHE):
        with open(REWARD_CACHE, 'rb') as fh:
            reward_results = pickle.load(fh)
        print(f'Loaded cached reward results from {REWARD_CACHE}')
    else:
        reward_results = {}
        for mode in ['pnl', 'sharpe', 'sortino']:
            reward_results[mode] = run_drl_wfo(returns, reward_mode=mode, timesteps=5_000)
        with open(REWARD_CACHE, 'wb') as fh:
            pickle.dump(reward_results, fh)
        print(f'Saved reward results to {REWARD_CACHE}')

    fig, ax = plt.subplots(figsize=(10, 4))
    for mode, r in reward_results.items():
        if len(r) > 0:
            (1 + r).cumprod().plot(ax=ax, label=f'Reward: {mode}')
    ax.set_title('DRL Performance by Reward Function Design')
    ax.set_ylabel('Cumulative Return')
    ax.legend()
    plt.tight_layout()
    plt.show()
else:
    print('Reward experiment skipped (SB3 not available or insufficient DRL results).')
    print('Key lesson: reward design is the most critical engineering decision in DRL trading.')
    print('  - pnl reward: maximises return, ignores risk')
    print('  - sharpe reward: maximises risk-adjusted return')
    print('  - sortino reward: maximises Sortino ratio (penalises only downside)')
    print('  - risk: all modes can be hacked if not constrained properly')

## Interpretation: Reward Function Design (Reward Hacking)

**Key problem:** The agent optimises the REWARD, not the GOAL.
If reward ≠ goal → the agent will find a way to "hack" the reward.

**Examples of reward hacking in portfolios:**
| Reward | Hacking risk | Fix |
|---------|-----------------|-----|
| Rolling Sharpe | Agent goes flat: rolling std → 0, reward = 0/0 = NaN → 0 | Penalty for going flat |
| Cumulative P&L | Agent takes enormous one-off risk | MD constraint |
| Sortino | Agent avoids every move (zero vol) | Minimum activity |

**Empirical reward comparison:**
- `pnl`: aggressive agent, high gains and large losses → high variance
- `sharpe`: more conservative agent → lower gains, but more stable
- `sortino`: intermediate → accepts gains, avoids losses

**Differential Sharpe (Moody & Saffell, 2001):**
A differential approximation of the Sharpe ratio → suitable for training via backprop:
```
DS_t = (B_t · ΔR_t − 0.5 · A_t · ΔR_t²) / (B_t − A_t²)^(3/2)
```
Where A_t, B_t = the current (recursive) moments.
Advantage: differentiable → it can be used directly as a loss in the network.

> **Conclusion:** There is no "ideal" reward. Choose it according to your acceptable risk,
> always test it with a circuit-breaker, and verify the agent's OOS behaviour.

## 9. Risk Management Module

Demonstrates circuit-breaker logic: halt trading when MD exceeds threshold.

In [ ]:
def apply_risk_circuit_breaker(strategy_returns: pd.Series,
                                mdd_threshold: float = -0.15,
                                cooldown_days: int = 20) -> pd.Series:
    """
    Hard circuit-breaker: go flat when rolling MD exceeds threshold.
    Re-engage after cooldown_days.
    """
    r = strategy_returns.copy()
    w = (1 + r).cumprod()
    dd = (w - w.cummax()) / w.cummax()

    result = r.copy()
    flat_until = -1

    for i, (date, d) in enumerate(dd.items()):
        if i <= flat_until:
            result.loc[date] = 0.0
        elif d < mdd_threshold:
            result.loc[date] = 0.0
            flat_until = i + cooldown_days

    return result


# Apply to best strategy (ML-Markowitz)
r_protected = apply_risk_circuit_breaker(r_ml_mw.fillna(0), mdd_threshold=-0.12)

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

ax = axes[0]
(1 + r_ml_mw.fillna(0)).cumprod().plot(ax=ax, color=COLORS['ml_mw'], label='ML+LW (no CB)')
(1 + r_protected).cumprod().plot(ax=ax, color='#228B22', label='ML+LW + Circuit Breaker', ls='--')
ax.set_title('Circuit Breaker Effect on ML-Markowitz')
ax.legend()

ax2 = axes[1]
for name, r, c in [('No CB', r_ml_mw.fillna(0), COLORS['ml_mw']),
                    ('With CB', r_protected, '#228B22')]:
    w  = (1 + r).cumprod()
    dd = (w - w.cummax()) / w.cummax()
    dd.plot(ax=ax2, label=name, color=c)
ax2.set_title('Drawdown Comparison')
ax2.legend()

plt.tight_layout()
plt.show()

print('\n--- Circuit Breaker Impact ---')
for name, r in [('Without CB', r_ml_mw.fillna(0)), ('With CB', r_protected)]:
    print(f'{name:<14}: Sharpe={sharpe(r):.2f}, MD={max_drawdown(r):.1%}, ARC={cagr(r):.1%}')

## Interpretation: Circuit Breaker — Protection Against Tail Risk

**What is a circuit breaker?**
A mechanism that automatically freezes the strategy when the MD exceeds a threshold:
```
if MD(t) < -threshold:
    order → cash for cooldown_days days
```

**Typical parameters:**
- `mdd_threshold = -0.12` (12%): aggressive strategy
- `mdd_threshold = -0.20` (20%): conservative strategy
- `cooldown_days = 20` (1 trading month): standard reset

**Circuit breaker effect (expected):**
- Sharpe: may drop slightly (we forgo part of the gain on false triggers)
- MD: significantly lower → the key benefit
- Calmar = ARC / MD: rises → a better return/max-risk ratio

**When is a circuit breaker indispensable?**
- DRL strategies: the agent may enter an unstable behavioural regime.
- ML strategies with GMADL: during a regime shift the model loses its edge.
- Short options: the loss can be unlimited without a CB.

**Additional safeguards in production:**
- `VIX-gated sizing`: reduce the position when VIX > 30
- `Concentration limit`: max 30% in a single asset
- `Live vs backtest divergence alert`: if the live SR < 50% of the historical SR → halt

---

## Summary — Session 3

**3 key lessons:**
1. **Classical Markowitz** needs ML to estimate μ and to regularise Σ in order to work in OOS.
2. **DRL (SAC)** learns a complex allocation policy → the greatest flexibility,
   but it requires careful design of the environment and the reward function.
3. **Risk management** (circuit breaker, VIX-gating) is mandatory → AI does not eliminate
   tail risk, it only changes its shape.

**Pipeline of a complete AI system:**
```
Data → Features → ML/DRL → MADL Filter → Risk Layer → Execution → Monitoring
```
Each layer is independently testable and replaceable.